# Preprocessing the MUDI Drug–Drug Interaction Dataset

This notebook preprocesses the MUDI (Multi-label Drug Interaction) dataset to prepare it for machine learning models that predict drug–drug interaction (DDI) types based on molecular structure.

Key Objectives

- Map drug identifiers to molecular structures (SMILES)

- Convert raw dataset into a structure-based representation

- Encode interaction types into numerical labels

- Prepare clean train and test datasets

- Create a combined dataset for downstream modeling

https://zenodo.org/records/15544552

In [16]:
!pip install rdkit

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import pandas as pd
import numpy as np
import json
import os
from rdkit import Chem
from rdkit.Chem import PandasTools

In [19]:
data_path = '/content/drive/MyDrive/FYP/IRP/Data/MUDI'
train_df = pd.read_csv(os.path.join(data_path, 'dataset/train.csv'))
test_df = pd.read_csv(os.path.join(data_path, 'dataset/test.csv'))


In [20]:
train_df['Interaction'].value_counts()

,count
Interaction,
Synergism,186268
Antagonism,27320
New Adverse,7527


In [21]:
len(train_df['Drug1'].unique())

989

In [22]:
import json

# Load the JSON file
with open('/content/drive/MyDrive/FYP/IRP/Data/MUDI/drug_info.json', 'r') as f:
    drug_info = json.load(f)

# Number of unique drug IDs (keys of the dictionary)
num_unique_drugs = len(drug_info)
print(f"Number of unique drug IDs in drug_info.json: {num_unique_drugs}")

Number of unique drug IDs in drug_info.json: 1295


## Create a Mapping from Drug ID to SMILES

The drug IDs in the CSV files have the format "Compound::DBxxxxx", while the keys in drug_info are simply "DBxxxxx". Need to strip the prefix.

In [23]:
# Function to clean drug ID
def clean_drug_id(drug_id):
    # Remove 'Compound::' prefix if present
    return drug_id.replace('Compound::', '')

# Build a dictionary that maps cleaned ID to SMILES
id_to_smiles = {}
for drug_id, info in drug_info.items():
    if 'smiles' in info:
        id_to_smiles[drug_id] = info['smiles']

print("Example mapping:")
print("DB00842 ->", id_to_smiles.get('DB00842', 'Not found'))

Example mapping:
DB00842 -> C1=CC=C(C=C1)C2=NC(C(=O)NC3=C2C=C(C=C3)Cl)O


## Map SMILES to Each Row

Create new columns smiles_a and smiles_b by applying the cleaning and mapping.

In [24]:
def get_smiles(row, drug_col):
    raw_id = row[drug_col]
    clean_id = clean_drug_id(raw_id)
    return id_to_smiles.get(clean_id, None)

# Apply to both dataframes
for df in [train_df, test_df]:
    df['smiles_a'] = df.apply(lambda row: get_smiles(row, 'Drug1'), axis=1)
    df['smiles_b'] = df.apply(lambda row: get_smiles(row, 'Drug2'), axis=1)

# Check for missing SMILES (should be none if all drugs are in drug_info)
print("Missing SMILES in train:", train_df['smiles_a'].isna().sum() + train_df['smiles_b'].isna().sum())
print("Missing SMILES in test:", test_df['smiles_a'].isna().sum() + test_df['smiles_b'].isna().sum())

Missing SMILES in train: 0
Missing SMILES in test: 0


In [25]:
# Canonicalize SMILES
def canonicalize_smiles(smiles):
    if smiles is None:
        return None
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True)

for df in [train_df, test_df]:
    df['smiles_a'] = df['smiles_a'].apply(canonicalize_smiles)
    df['smiles_b'] = df['smiles_b'].apply(canonicalize_smiles)

print("Canonicalization complete.")

Canonicalization complete.


In [26]:
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

train shape: (221115, 5)
test shape: (89417, 5)


In [27]:
# Remove duplicates based on canonical SMILES pairs and label
for df in [train_df, test_df]:
    # Optional: sort the pair so that Drug1 and Drug2 order doesn't matter
    df[['smiles_a', 'smiles_b']] = np.sort(df[['smiles_a', 'smiles_b']], axis=1)
    df.drop_duplicates(subset=['smiles_a', 'smiles_b', 'Interaction'], inplace=True)

print("Duplicates removed.")
print("New train shape:", train_df.shape)
print("New test shape:", test_df.shape)

Duplicates removed.
New train shape: (175045, 5)
New test shape: (69876, 5)


## Encode Interaction Labels


The dataset contains three interaction types:

- Synergism

- Antagonism

- New Effect (New Adverse)

Will map them to numeric values for classification.

In [28]:
# Define mapping
interaction_map = {
    'Synergism': 0,
    'Antagonism': 1,
    'New Adverse': 2
}

# Apply mapping
train_df['label'] = train_df['Interaction'].map(interaction_map)
test_df['label'] = test_df['Interaction'].map(interaction_map)

# Verify distribution
print("Train label distribution:\n", train_df['label'].value_counts())
print("\nTest label distribution:\n", test_df['label'].value_counts())

Train label distribution:
 label
0    140198
1     27320
2      7527
Name: count, dtype: int64

Test label distribution:
 label
0    54711
1    11914
2     3251
Name: count, dtype: int64


In [29]:
# Select and save train data
train_processed = train_df[['smiles_a', 'smiles_b', 'label']].copy()
train_processed.to_csv(os.path.join(data_path, 'train_processed.csv'), index=False)

# Select and save test data
test_processed = test_df[['smiles_a', 'smiles_b', 'label']].copy()
test_processed.to_csv(os.path.join(data_path, 'test_processed.csv'), index=False)

print("Processed files saved:")
print("- train_processed.csv")
print("- test_processed.csv")

Processed files saved:
- train_processed.csv
- test_processed.csv


In [30]:
train_processed.sample(5)

,smiles_a,smiles_b,label
116772,CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O,Nc1nc(N)c2nc(-c3ccccc3)c(N)nc2n1,0
26584,CC(C)c1cccc(C(C)C)c1OCOP(=O)(O)O,CN(C)CC/C=C1/c2ccccc2COc2ccccc21,0
97237,C[C@@H]1CC(=O)NN=C1c1ccc(NN=C(C#N)C#N)cc1,Cc1nc2n(c(=O)c1CCN1CCC(c3noc4cc(F)ccc34)CC1)CCCC2,0
182392,CC(=O)Oc1cc2c(s1)CCN(C(C(=O)C1CC1)c1ccccc1F)C2,NCCc1ccc(O)c(O)c1,0
102237,CC#CCn1c(N2CCC[C@@H](N)C2)nc2c1c(=O)n(Cc1nc(C)...,Cc1nnc(NS(=O)(=O)c2ccc(N)cc2)s1,0


In [31]:
print(f"Number of duplicate rows in train_df: {train_df.duplicated().sum()}")
print(f"Number of duplicate rows in test_df: {test_df.duplicated().sum()}")

Number of duplicate rows in train_df: 0
Number of duplicate rows in test_df: 0


In [32]:
train_processed.shape, test_processed.shape

((175045, 3), (69876, 3))

## Merge train + test

In [33]:
full_dataset = pd.concat([train_processed, test_processed], axis=0, ignore_index=True)

# save to CSV
# full_dataset.to_csv(os.path.join(data_path, 'MUDI_full_dataset_processed.csv'), index=False)

print(f"Full dataset shape: {full_dataset.shape}")
print(f"Label distribution:\n{full_dataset['label'].value_counts()}")

Full dataset shape: (244921, 3)
Label distribution:
label
0    194909
1     39234
2     10778
Name: count, dtype: int64


In [34]:
print(f"Number of duplicate rows in full dataset: {full_dataset.duplicated().sum()}")

Number of duplicate rows in full dataset: 0


In [35]:
full_dataset.sample(5)

,smiles_a,smiles_b,label
217001,CCCNC(C)C(=O)Nc1ccccc1C,C[C@]12CCC(=O)C=C1CC[C@@H]1[C@@H]2CC[C@@]2(C)[...,0
177843,CO/N=C(\C(=O)N[C@@H]1C(=O)N2C(C(=O)O)=C(CSc3nc...,Cc1ccc(Cl)c(Nc2ccccc2C(=O)O)c1Cl,0
231277,CN(C)c1nc(N(C)C)nc(N(C)C)n1,COc1ccc(-n2nc(C(N)=O)c3c2C(=O)N(c2ccc(N4CCCCC4...,0
34302,NCCc1ccc(O)c(O)c1,O=[N+]([O-])OCC(CO[N+](=O)[O-])O[N+](=O)[O-],1
119546,CC(=O)CC(c1ccccc1)c1c(O)c2ccccc2oc1=O,COc1cc(C)nc(-n2nc(C)cc2OC)n1,0


In [36]:
full_dataset.shape

(244921, 3)